# Forecasting

**Objective:** Forecast reserve money, currency in circulation, and net foreign exchange assets, then evaluate model accuracy.


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from rbi_utils import *

sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_columns', 120)


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.statespace.sarimax import SARIMAX

df = load_cleaned().set_index('date').asfreq('W-FRI')
series = df['reserve_money'].interpolate()
train = series.iloc[:-52]
test = series.iloc[-52:]

In [ ]:
model = SARIMAX(train, order=(1, 1, 1), seasonal_order=(1, 1, 1, 52), enforce_stationarity=False, enforce_invertibility=False)
result = model.fit(disp=False)
forecast = result.get_forecast(steps=len(test))
pred = forecast.predicted_mean
rmse = mean_squared_error(test, pred, squared=False)
mae = mean_absolute_error(test, pred)
print({'RMSE': rmse, 'MAE': mae, 'MAPE': mape(test, pred)})

In [ ]:
future = result.get_forecast(steps=26)
forecast_table = future.summary_frame().reset_index().rename(columns={'index': 'date'})
forecast_table.to_csv(PROJECT_ROOT / 'data' / 'processed' / 'reserve_money_forecast.csv', index=False)
forecast_table.head()

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
series.tail(180).plot(ax=ax, label='Historical')
pred.plot(ax=ax, label='Test Forecast')
future.predicted_mean.plot(ax=ax, label='Future Forecast')
ax.legend()
ax.set_title('Reserve Money Forecast')
save_figure(fig, '08_reserve_money_forecast.png')